##### Copyright 2025 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Embeddings with OpenAI SDK and Gemini

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/openai/Embeddings.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

This notebook demonstrates how to generate **text embeddings** using the **Gemini embedding models** via the **OpenAI SDK**.

Embeddings are numerical representations of text that capture semantic meaning. They're useful for:
- Semantic search
- Document similarity
- Clustering and classification
- Recommendation systems

## Setup

In [ ]:
%pip install -U -q openai numpy scikit-learn

In [ ]:
from openai import OpenAI
import os
import numpy as np

try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
except:
    GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY', '--enter-your-API-key-here--')

client = OpenAI(
    api_key=GOOGLE_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

## Generate a Single Embedding

Let's start by generating an embedding for a single piece of text:

In [ ]:
text = "The quick brown fox jumps over the lazy dog."

response = client.embeddings.create(
    model="text-embedding-004",
    input=text
)

embedding = response.data[0].embedding

print(f"Text: {text}")
print(f"Embedding dimension: {len(embedding)}")
print(f"First 10 values: {embedding[:10]}")

## Batch Embeddings

Generate embeddings for multiple texts at once:

In [ ]:
texts = [
    "I love programming in Python.",
    "Machine learning is fascinating.",
    "The weather is beautiful today.",
    "Artificial intelligence is transforming technology."
]

response = client.embeddings.create(
    model="text-embedding-004",
    input=texts
)

embeddings = [item.embedding for item in response.data]

print(f"Generated {len(embeddings)} embeddings")
print(f"Each embedding has {len(embeddings[0])} dimensions")
print(f"\nUsage: {response.usage.total_tokens} tokens")

## Semantic Similarity

Calculate similarity between texts using cosine similarity:

In [ ]:
def cosine_similarity(a, b):
    """Calculate cosine similarity between two vectors"""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Compare similarities
print("Similarity scores:")
print("-" * 60)

for i in range(len(texts)):
    for j in range(i + 1, len(texts)):
        similarity = cosine_similarity(embeddings[i], embeddings[j])
        print(f"'{texts[i]}'")
        print(f"  vs")
        print(f"'{texts[j]}'")
        print(f"  Similarity: {similarity:.4f}")
        print()

## Semantic Search Example

Find the most similar documents to a query:

In [ ]:
# Document collection
documents = [
    "Python is a high-level programming language.",
    "Machine learning models can predict outcomes.",
    "The sun is shining brightly today.",
    "Neural networks are inspired by the human brain.",
    "I enjoy hiking in the mountains.",
    "Deep learning is a subset of machine learning.",
    "The ocean waves are calming.",
    "JavaScript is popular for web development."
]

# Generate embeddings for documents
doc_response = client.embeddings.create(
    model="text-embedding-004",
    input=documents
)
doc_embeddings = [item.embedding for item in doc_response.data]

# Search query
query = "Tell me about artificial intelligence"

# Generate query embedding
query_response = client.embeddings.create(
    model="text-embedding-004",
    input=query
)
query_embedding = query_response.data[0].embedding

# Calculate similarities
similarities = [
    cosine_similarity(query_embedding, doc_emb)
    for doc_emb in doc_embeddings
]

# Sort by similarity
results = sorted(
    zip(documents, similarities),
    key=lambda x: x[1],
    reverse=True
)

print(f"Query: '{query}'")
print("\nTop 3 most relevant documents:")
print("=" * 60)
for i, (doc, score) in enumerate(results[:3], 1):
    print(f"{i}. {doc}")
    print(f"   Similarity: {score:.4f}")
    print()

## Document Clustering

Use embeddings to cluster similar documents:

In [ ]:
from sklearn.cluster import KMeans

# Example documents about different topics
docs = [
    # Technology
    "Python programming is powerful",
    "JavaScript frameworks are popular",
    "Machine learning algorithms learn from data",
    
    # Nature
    "Mountains are majestic",
    "Ocean waves are soothing",
    "Forests are full of life",
    
    # Food
    "Pizza is delicious",
    "Sushi is a Japanese delicacy",
    "Pasta comes in many shapes"
]

# Generate embeddings
response = client.embeddings.create(
    model="text-embedding-004",
    input=docs
)
embeddings_matrix = np.array([item.embedding for item in response.data])

# Cluster into 3 groups
kmeans = KMeans(n_clusters=3, random_state=42)
clusters = kmeans.fit_predict(embeddings_matrix)

# Display results
print("Document clusters:")
print("=" * 60)
for cluster_id in range(3):
    print(f"\nCluster {cluster_id + 1}:")
    cluster_docs = [doc for doc, c in zip(docs, clusters) if c == cluster_id]
    for doc in cluster_docs:
        print(f"  - {doc}")

## Task-Specific Embeddings

Gemini embeddings support different task types for optimized performance:

In [ ]:
# Note: Task types are specified differently in Gemini SDK
# With OpenAI SDK, use the default which is optimized for general use

# For semantic search, retrieval, and similarity:
query = "What is machine learning?"
documents = [
    "Machine learning is a subset of AI",
    "The sky is blue",
    "Neural networks are used in ML"
]

# Generate embeddings
all_texts = [query] + documents
response = client.embeddings.create(
    model="text-embedding-004",
    input=all_texts
)

query_emb = response.data[0].embedding
doc_embs = [item.embedding for item in response.data[1:]]

# Find most similar
similarities = [cosine_similarity(query_emb, doc_emb) for doc_emb in doc_embs]
best_match_idx = np.argmax(similarities)

print(f"Query: {query}")
print(f"Best match: {documents[best_match_idx]}")
print(f"Similarity: {similarities[best_match_idx]:.4f}")

## Embedding Dimensions

Gemini's text-embedding-004 model produces 768-dimensional embeddings:

In [ ]:
response = client.embeddings.create(
    model="text-embedding-004",
    input="Sample text"
)

embedding = response.data[0].embedding
print(f"Embedding dimensions: {len(embedding)}")
print(f"Embedding type: {type(embedding)}")
print(f"Value range: [{min(embedding):.4f}, {max(embedding):.4f}]")

## Cost and Performance

Check token usage for embeddings:

In [ ]:
texts = [
    "Short text",
    "This is a medium length text with more words to embed.",
    "This is a much longer text that contains significantly more content and will use more tokens when generating embeddings. It demonstrates how token usage scales with input length."
]

for text in texts:
    response = client.embeddings.create(
        model="text-embedding-004",
        input=text
    )
    
    print(f"Text length: {len(text)} characters")
    print(f"Tokens used: {response.usage.total_tokens}")
    print()

## Practical Applications

### 1. Question Answering System

In [ ]:
# Knowledge base
kb = [
    "Gemini is Google's most capable AI model",
    "The Gemini API supports text, images, audio, and video",
    "You can use Gemini with the OpenAI SDK",
    "Gemini models include Flash, Pro, and Nano variants"
]

# Embed knowledge base
kb_response = client.embeddings.create(
    model="text-embedding-004",
    input=kb
)
kb_embeddings = [item.embedding for item in kb_response.data]

def answer_question(question):
    # Embed question
    q_response = client.embeddings.create(
        model="text-embedding-004",
        input=question
    )
    q_embedding = q_response.data[0].embedding
    
    # Find most relevant
    similarities = [cosine_similarity(q_embedding, kb_emb) for kb_emb in kb_embeddings]
    best_idx = np.argmax(similarities)
    
    return kb[best_idx], similarities[best_idx]

# Test
questions = [
    "What is Gemini?",
    "What formats does Gemini support?",
    "Can I use OpenAI SDK with Gemini?"
]

for q in questions:
    answer, score = answer_question(q)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print(f"Confidence: {score:.4f}")
    print()

## Best Practices

1. **Batch Processing**: Generate embeddings in batches for better performance
2. **Caching**: Cache embeddings for frequently used texts
3. **Normalization**: Embeddings are already normalized for cosine similarity
4. **Model Selection**: Use `text-embedding-004` for best quality
5. **Token Limits**: Be aware of input token limits per request

## Comparison: OpenAI SDK vs Gemini SDK

**OpenAI SDK (this notebook):**
```python
response = client.embeddings.create(
    model="text-embedding-004",
    input=text
)
embedding = response.data[0].embedding
```

**Gemini SDK:**
```python
import google.generativeai as genai

result = genai.embed_content(
    model="models/text-embedding-004",
    content=text,
    task_type="retrieval_document"  # Additional optimization
)
embedding = result['embedding']
```

Both approaches work well! The Gemini SDK offers task-specific optimizations.

## Next Steps

- [Structured Outputs](./Structured_Outputs.ipynb) - Extract structured data
- [Basic Chat](./Basic_Chat.ipynb) - Text generation examples
- [Full OpenAI Compatibility Guide](../Get_started_OpenAI_Compatibility.ipynb)
- [Gemini Embeddings Guide](../Embeddings.ipynb) - More embedding examples